# Notebook 4 — Modello Transformer

Obiettivo: addestrare e valutare il Transformer su tutti e 4 i sotto-dataset CMAPSS, con la stessa procedura dell'LSTM per un confronto fair.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.models.transformer_model import build_transformer
from src.training import run_experiment

DATA_DIR = '../data/processed'
MODEL_DIR = '../saved_models'

## Architettura Transformer

```
Input (30, 17)
    │
Dense(64)                          ← proiezione lineare: 17 sensori → spazio d_model
    │
PositionalEncoding (sinusoidale)   ← aggiunge info sulla posizione temporale
    │
TransformerEncoderBlock × 2
  [MultiHeadAttention(4 head) → Add&Norm → FFN(128→64) → Add&Norm]
    │
GlobalAveragePooling1D             ← aggrega tutti i 30 timestep con media
    │
Dense(64, relu)
    │
Dropout(0.1)
    │
Dense(1, linear)                   ← predizione RUL
```

**Scelte architetturali:**
- **d_model=64**: dimensione interna del Transformer (spazio di embedding)
- **num_heads=4**: 4 attention head paralleli, ognuno apprende relazioni diverse tra i timestep. `key_dim = 64/4 = 16` per head
- **ff_dim=128**: dimensione della FFN = 2× d_model, standard in letteratura (Vaswani et al., 2017)
- **num_blocks=2**: 2 blocchi encoder impilati, sufficiente per sequenze brevi (30 timestep)
- **PE sinusoidale**: nessun parametro apprendibile → più stabile, formula: `PE(pos,2i)=sin(pos/10000^(2i/d))`
- **GlobalAveragePooling1D** invece di usare solo l'ultimo token: sfrutta tutta la sequenza, più stabile per la regressione

In [ ]:
results = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    X_train = np.load(f'{DATA_DIR}/X_train_{fd}.npy')
    y_train = np.load(f'{DATA_DIR}/y_train_{fd}.npy')
    X_test  = np.load(f'{DATA_DIR}/X_test_{fd}.npy')
    y_test  = np.load(f'{DATA_DIR}/y_test_{fd}.npy')

    window_size = X_train.shape[1]  # 30
    n_features  = X_train.shape[2]  # 17

    # Costruisco il Transformer con gli iperparametri scelti
    model = build_transformer(
        window_size, n_features,
        d_model=64,    # dimensione embedding
        num_heads=4,   # 4 head di attention paralleli
        ff_dim=128,    # FFN interna = 2× d_model
        num_blocks=2   # 2 blocchi encoder
    )
    model_path = f'{MODEL_DIR}/transformer_{fd}.keras'

    # Stessa procedura dell'LSTM → confronto fair: stessi dati, stessi callback, stesso test
    res = run_experiment(model, X_train, y_train, X_test, y_test, model_path)
    results[fd] = res
    print(f'{fd}: {res["metrics"]}')

## Analisi delle curve di training

Il Transformer converge **molto più rapidamente** dell'LSTM (poche epoche vs decine):
il meccanismo di self-attention cattura immediatamente le dipendenze globali tra i timestep,
senza il processo graduale di "scoperta" tipico delle LSTM.

**FD001, FD002, FD003**: curve lisce e allineate tra train e val → nessun overfitting.

**FD004** (il più complesso: 6 condizioni + 2 modi di guasto): si osserva una lieve
oscillazione nella fase finale, con un piccolo divario train/val → il Transformer
fatica a generalizzare su distribuzioni non stazionarie. Questo si riflette nell'RMSE
finale (39.4 vs 28.7 dell'LSTM).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (fd, res) in zip(axes.flatten(), results.items()):
    ax.plot(res['history']['loss'], label='train')
    ax.plot(res['history']['val_loss'], label='val')
    ax.set_title(f'Transformer — {fd}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    # Convergenza rapida in poche epoche: attention cattura subito le dipendenze globali
    ax.legend()
plt.tight_layout()
plt.savefig('../plots/04_transformer_training_curves.png', dpi=150)
plt.show()